# clustering_001  –  KIER KMeans 군집화 파이프라인
`KMeans_Find K` 최적 K 탐색 → KMeans 군집화 → `Integration by labels` 클러스터별 통합

## 01. Init

In [ ]:
#region Import_Basic
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
pd.options.display.float_format = '{:.10f}'.format

import random
import datetime as dt
from datetime import datetime, date, timedelta

import matplotlib.pyplot as plt, seaborn as sns
plt.rcParams['figure.figsize'] = [10, 8]

from tqdm.notebook import tqdm
#endregion

In [ ]:
from core import (data_datetime as com_date, data_clustering as com_clustering,
                  provider_kier_m02 as com_KIER_M02)
from core.ref_cluster_eval import get_dunn_index
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

## 02. 최적 K 탐색 (Elbow / CHI / Silhouette / Dunn)

 - CODE : KIER M02 - Clustering
 - DESC  
    &ensp; : 최적의 Cluster를 선정하기 위한 정량적 비교  
    &emsp; 1) Inertia 기반의 Elbow-Method  
    &emsp; 2) 군집화 계수 비교 : Silhouette / CHI / Dunn Index  

  - DATE  
    &ensp; 2024-02-01 Created  
    &ensp; 2024-04-03 코드 개선  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; 1) KIER M02 초기부분 공통코드화  
    &ensp; 2024-04-04 Updated  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; 1) 기능 구현 완료 및 논문 작성    
    &ensp; 2024-07-23 Updated  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; 1) Dunn Index 부분 추가    

In [ ]:
#region Import_Basic
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
pd.options.display.float_format = '{:.10f}'.format

import random
import datetime as dt
from datetime import datetime, date, timedelta

import matplotlib.pyplot as plt, seaborn as sns
plt.rcParams['figure.figsize'] = [10, 8]

from tqdm.notebook import tqdm
#endregion

In [ ]:
# DL 라이브러리 (클러스터링 노트북 – 불필요)
# 모델링 노트북(M02-03_*)에서만 사용

In [ ]:
## Import_Local
from core import data_datetime as com_date, provider_kma as com_KMA, provider_keco as com_KECO, provider_kasi as com_Holi, provider_kier_m02 as com_KIER_M02, data_clustering as com_clustering

In [ ]:
## Dict_Domain
## {0:"ELEC", 1:"HEAT", 2:"WATER", 3:"HOT_HEAT", 4:"HOT_FLOW", 99:"GAS"}
## {'10min', '1H', '1D', '1W', '1M'}
int_domain, int_interval = 0, 2

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)
## Interval, Target File
str_interval, str_fileRaw, str_fileRaw_hList, str_file = com_KIER_M02.create_file_str(str_domain, int_interval)

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

In [ ]:
str_file = str('KIER_' + str_domain + '_INST_03_IQR.csv')
df_kier_raw = pd.read_csv(str_dirName_h + str_file, index_col = 0)
df_kier_raw['METER_DATE'] = pd.to_datetime(df_kier_raw['METER_DATE'])
print(df_kier_raw.isna().sum().sum())
df_kier_raw

In [ ]:
## 호실별 순시 사용량 컬럼만 가져오기
list_col_tar = list(df_kier_raw.columns[6:-2])
df_kier_h = df_kier_raw.set_index('METER_DATE')
df_kier_h

In [ ]:
# ## Error Log : "[5:-2]" 부분을 추가하여 연월일시 및 평균합계 부분을 제거해주지 않으면, 군집화 계수가 제대로 도출되지 못함.
# df_kier_summary_total = df_kier_h.transpose().reset_index()[5:-2]
# ## 또는, 가장 깔끔하게 이렇게 처리해도 좋다
df_kier_summary_total = df_kier_h[list_col_tar].transpose().reset_index()

## 세대 번호의 컬럼명이 'index'로 지정되어 오류 발생
df_kier_summary_total['h_index'] = df_kier_summary_total['index']
df_kier_summary_total = df_kier_summary_total.drop(columns = ['index'])
df_kier_summary_total

In [ ]:
X = df_kier_summary_total.drop(columns = 'h_index')
y = df_kier_summary_total['h_index']
X.isna().sum()
# y

http://bigdata.dongguk.ac.kr/lectures/datascience/_book/%EA%B5%B0%EC%A7%91%EB%B6%84%EC%84%9D.html

In [ ]:
# 변수 표준화
scaler = StandardScaler() # 변수 표준화 클래스
scaler.fit(X)  # 표준화를 위해 변수별 파라미터(평균, 표준편차) 계산
# scaler.mean_, scaler.scale_
X_std = scaler.transform(X)  # 훈련자료 표준화 변환
# X_std

### Clustering (군집화) : K - Means

군집의 수 결정 방법  
1) elbow method - 군집의 개수와 군집내 변동의 합을 그래프로 나타내고, 변동량의 변화가 작아지는 지점의 군집의 수를 적정 군집의 수로 결정함  
2) 군집화시 계수 비교 : Silhouette / CHI

In [ ]:
## 군집시 군집의 수 판단을 위한 Data 수집, 이를 바탕으로 인사이트 도출
int_cluster_min, int_cluster_max = 2, 10 ## 최소 / 최대 군집 수

In [ ]:
list_intertia, list_intertia_deriv = com_clustering.clustering_elbow_method(str_interval, int_cluster_min, int_cluster_max, X_std
                                                                            , opt_X = 3)
print(list_intertia)
print(list_intertia_deriv)

In [ ]:
list_CHI = com_clustering.clustering_CHI_method(str_interval, int_cluster_min, int_cluster_max, X_std, 2)
print(list_CHI)

In [ ]:
list_Silhouette, list_cnt_clusters_by_K = com_clustering.clustering_silhouette_method(str_interval, int_cluster_min, int_cluster_max, X_std, 2)
print(list_Silhouette)
print(list_cnt_clusters_by_K)

In [ ]:
from core.ref_cluster_eval import get_dunn_index

opt_X = 2

## 예외처리01
## Min이 Max보다 크면 그냥 바꿔줌 + Int가 아니면 Int로 바꿔줌
if int_cluster_min > int_cluster_max : int_clusters_min, int_clusters_max = int(int_cluster_max), int(int_cluster_min) + 1
else : int_clusters_min, int_clusters_max = int(int_cluster_min), int(int_cluster_max) + 1

## 초기 변수  생성
list_Dunn = []
K = range(int_clusters_min, int_clusters_max)

for n_cluster in K:
    km_dunn = KMeans(n_clusters = n_cluster
                    , init="k-means++"
                    , max_iter=300
                    , n_init=1).fit(X_std) 
    cluster = km_dunn.predict(X_std)
    list_Dunn.append(get_dunn_index(X_std, cluster))

fig = plt.figure(figsize=(10,10))
fig.set_facecolor('white')
ax = fig.add_subplot()
ax.plot(K, list_Dunn, marker='.', markersize = 5, zorder = 2)
if opt_X != None : plt.scatter(opt_X, list_Dunn[opt_X - 2], color = 'red', marker = '^', label = 'Point', zorder = 9999)
ax.set_xticks(K)
plt.xlabel('k')
plt.ylabel('Dunn Index')
plt.title('Dunn Index by number of clusters (Interval : ' + str_interval + ')')
plt.show()

list_Dunn

In [ ]:
print(end)

In [ ]:
## 위에서 결정된 군집의 수에 따라 군집화 결과 도출
## 초기 변수 생성
K, cnt_loop = 2, 10 ## K : 결정된/평가할 군집의 수, cnt_loop : 평가를 위한 군집화 시도 횟수

km = KMeans(n_clusters = K, init="k-means++", max_iter=300, n_init=1).fit(X_std)
cluster = km.predict(X_std)

list_log_clusters = com_clustering.clustering_get_cnt_by_loop(K, cnt_loop, X_std)
print("총 " + str(cnt_loop) + "회에 걸친 군집화 시뮬레이션")
print(list_log_clusters)

In [ ]:
## 최종 군집에 대한 군집화 평가
print(com_clustering.get_cluster_sizes(km, X_std)) ## 최종 군집화에 대한 군집 크기 출력

com_clustering.clustering_visualization(str_interval, km, X_std)
list_scores = com_clustering.get_clustering_score(km, X_std, y)

In [ ]:
## 최종 군집에 대한 Labeled Data 저장
print(com_clustering.get_cluster_sizes(km, X_std)) ## 최종 군집화에 대한 군집 크기 출력
# df_kier_summary_total['target'] = np.transpose(np.where(km.labels_ == i)[0])
df_kier_summary_total['target_'+str_domain] = 0
for i in range(0, len(df_kier_summary_total)) : df_kier_summary_total['target_'+str_domain].iloc[i] = km.labels_[i]
# df_kier_summary_total[['h_index', 'target_' + str_domain]]

str_file_labeled = str_dirName_h + 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'
df_kier_summary_total = df_kier_summary_total[['h_index', 'target_'+str_domain]]
df_kier_summary_total.to_csv(str_file_labeled)
print(str_file_labeled)
df_kier_summary_total

## 03. 클러스터별 통합 데이터셋 생성

 - CODE : Clustering - 라벨에 따른 사용량 분리 및 저장  
 - DESC : 군집화 - 단지 전체의 에너지 사용량을 Cluster별로 분류하여 CSV 저장  
 - DATE  
    &ensp; 2024-02-22 Created  
    &ensp; 2024-02-25 Updated : 날씨 데이터 Merge 부분을 분리 및 제거 (intergration_Weather)  
    &ensp; 2024-08-05 Updated : Error Fix  

In [ ]:
#region Import_Basic
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
pd.options.display.float_format = '{:.10f}'.format

import random
import datetime as dt
from datetime import datetime, date, timedelta

import matplotlib.pyplot as plt, seaborn as sns
plt.rcParams['figure.figsize'] = [10, 8]

from tqdm.notebook import tqdm
#endregion

In [ ]:
# DL 라이브러리 (클러스터링 노트북 – 불필요)
# 모델링 노트북(M02-03_*)에서만 사용

In [ ]:
## Import_Local
from core import data_datetime as com_date, provider_kma as com_KMA, provider_keco as com_KECO, provider_kasi as com_Holi, provider_kier_m02 as com_KIER_M02

In [ ]:
## Define Todate str
_now = dt.datetime.now()
str_now_ymd = _now.date()
str_now_y, str_now_m, str_now_d = _now.year, _now.month, _now.day
str_now_hr, str_now_min = _now.hour, _now.minute

print(_now)
print(f"{str_now_y} / {str_now_m} / {str_now_d}")
print(f"{str_now_hr} : {str_now_min}")

In [ ]:
## Dict_Domain
## {0:"ELEC", 1:"HEAT", 2:"WATER", 3:"HOT_HEAT", 4:"HOT_FLOW", 99:"GAS"}
## {'10MIN', '1H', '1D', '1W', '1M'}
int_domain, int_interval = 0, 0

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)
## Interval, Target File
str_interval, str_fileRaw, str_fileRaw_hList, str_file = com_KIER_M02.create_file_str(str_domain, int_interval)

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

In [ ]:
str_file = 'KIER_' + str_domain + '_Input_Comb_' + str_interval + '.csv'
df_kier_raw = pd.read_csv(str_dirName_h + str_file, index_col = 0)
df_kier_raw

In [ ]:
df_kier_raw.columns[25:]

In [ ]:
## Column List
list_col_date = df_kier_raw.columns[:6].to_list() ## Date Data
list_col_weather = df_kier_raw.columns[7:23].to_list() ## Weather Data
df_kier_raw_weather = df_kier_raw[list_col_weather]
df_kier_raw_weather.insert(0, 'METER_DATE', df_kier_raw['METER_DATE'])

## Usage Data
list_kier_h_all = df_kier_raw.columns[28:].to_list()

print(list_col_date)
print(list_col_weather)
print(list_kier_h_all)

In [ ]:
## 직전 시간의 에너지 사용 Column
df_kier_raw_prev = df_kier_raw[list_col_date + list_kier_h_all]
df_kier_raw_prev['METER_DATE'] = df_kier_raw_prev['METER_DATE'].shift(-1)
df_kier_raw_prev = df_kier_raw_prev[:-1]

In [ ]:
## Label별 호실 List Load (From "M02-02_Clustering_02_KMeans_Find K.ipynb")
K = 2
str_file_clustering = 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'

df_kier_h_cluster = pd.read_csv(str_dirName_h + str_file_clustering, index_col = 0).rename(columns = {'index' : 'h_index'})[['h_index', 'target_' + str_domain]]
list_kier_h_c0 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 0]['h_index']
print(len(list_kier_h_c0))
list_kier_h_c1 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 1]['h_index']
print(len(list_kier_h_c1))
list_kier_h_c2 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 2]['h_index']
print(len(list_kier_h_c2))

In [ ]:
## 전체 사용량 합계
df_kier_h_all = df_kier_raw_weather.copy()
df_kier_h_tmp = df_kier_raw[list_kier_h_all]
df_kier_h_all['ELEC_INST_SUM'] = df_kier_h_tmp.sum(axis = 1)
df_kier_h_tmp['METER_DATE'] = df_kier_raw_weather['METER_DATE']
df_kier_h_all = pd.merge(df_kier_h_all, df_kier_raw_prev, how = 'left', on = 'METER_DATE').dropna()

## Cluster별 사용량 합계
## ■ C00
df_kier_h_c0 = df_kier_raw_weather.copy()
df_kier_h_tmp = df_kier_raw[list_kier_h_c0]
df_kier_h_c0['ELEC_INST_SUM_C0'] = df_kier_h_tmp.sum(axis = 1)
df_kier_h_tmp['METER_DATE'] = df_kier_raw_weather['METER_DATE']
df_kier_h_c0['ELEC_INST_SUM_C0'] = df_kier_h_tmp.sum(axis = 1)
df_kier_h_c0 = pd.merge(df_kier_h_c0, df_kier_raw_prev, how = 'left', on = 'METER_DATE').dropna()

# ## ■ C01
df_kier_h_c1 = df_kier_raw_weather.copy()
df_kier_h_tmp = df_kier_raw[list_kier_h_c1]
df_kier_h_c1['ELEC_INST_SUM_C1'] = df_kier_h_tmp.sum(axis = 1)
df_kier_h_tmp['METER_DATE'] = df_kier_raw_weather['METER_DATE']
df_kier_h_c1['ELEC_INST_SUM_C1'] = df_kier_h_tmp.sum(axis = 1)
df_kier_h_c1 = pd.merge(df_kier_h_c1, df_kier_raw_prev, how = 'left', on = 'METER_DATE').dropna()

# ## ■ C02
df_kier_h_c2 = df_kier_raw_weather.copy()
df_kier_h_tmp = df_kier_raw[list_kier_h_c2]
df_kier_h_c2['ELEC_INST_SUM_C2'] = df_kier_h_tmp.sum(axis = 1)
df_kier_h_tmp['METER_DATE'] = df_kier_raw_weather['METER_DATE']
df_kier_h_c2['ELEC_INST_SUM_C2'] = df_kier_h_tmp.sum(axis = 1)
df_kier_h_c2 = pd.merge(df_kier_h_c2, df_kier_raw_prev, how = 'left', on = 'METER_DATE').dropna()

df_kier_h_c2

In [ ]:
# ## 전체 사용량 합계
# df_kier_h_all = df_kier_raw[list_kier_h_all]
# df_kier_h_all['ELEC_INST_SUM'] = df_kier_h_all.sum(axis = 1)
# df_kier_h_all.insert(0, 'METER_DATE', df_kier_raw['METER_DATE'])
# df_kier_h_all = pd.merge(df_kier_raw_weather, df_kier_h_all
#                          , how = 'left', on = ['METER_DATE'])
# list_kier_h_all = ['METER_DATE'] + df_kier_raw_weather + ['ELEC_INST_SUM']
# df_kier_h_all = df_kier_h_all[list_kier_h_all]

# ## Cluster별 사용량 합계
# ## ■ C00
# df_kier_h_c0 = df_kier_raw[list_kier_h_c0]
# df_kier_h_c0['ELEC_INST_SUM_C0'] = df_kier_h_c0.sum(axis = 1)
# df_kier_h_c0.insert(0, 'METER_DATE', df_kier_raw['METER_DATE'])
# ## 합계행만 남기기 (필요시 세대별 사용량 복원 가능)
# df_kier_h_c0 = df_kier_h_c0[['METER_DATE', 'ELEC_INST_SUM_C0']]

# ## ■ C01
# df_kier_h_c1 = df_kier_raw[list_kier_h_c1]
# df_kier_h_c1['ELEC_INST_SUM_C1'] = df_kier_h_c1.sum(axis = 1)
# df_kier_h_c1.insert(0, 'METER_DATE', df_kier_raw['METER_DATE'])
# ## 합계행만 남기기 (필요시 세대별 사용량 복원 가능)
# df_kier_h_c1 = df_kier_h_c1[['METER_DATE', 'ELEC_INST_SUM_C1']]

# ## ■ C02
# df_kier_h_c2 = df_kier_raw[list_kier_h_c2]
# df_kier_h_c2['ELEC_INST_SUM_C2'] = df_kier_h_c2.sum(axis = 1)
# df_kier_h_c2.insert(0, 'METER_DATE', df_kier_raw['METER_DATE'])
# ## 합계행만 남기기 (필요시 세대별 사용량 복원 가능)
# df_kier_h_c2 = df_kier_h_c2[['METER_DATE', 'ELEC_INST_SUM_C2']]

In [ ]:
df_kier_h_all

In [ ]:
str_file = 'KIER_' + str_domain + '_INST_Weather_ALL_' + str_interval + '.csv'
df_kier_h_all.to_csv(str_dirName_h + str_file)
print(df_kier_h_all.info())
df_kier_h_all

In [ ]:
str_file = 'KIER_' + str_domain + '_INST_Weather_C0_' + str_interval + '.csv'
df_kier_h_c0.to_csv(str_dirName_h + str_file)
print(df_kier_h_c0.info())
df_kier_h_c0

In [ ]:
str_file = 'KIER_' + str_domain + '_INST_Weather_C1_' + str_interval + '.csv'
df_kier_h_c1.to_csv(str_dirName_h + str_file)
print(df_kier_h_c1.info())
df_kier_h_c1

In [ ]:
str_file = 'KIER_' + str_domain + '_INST_Weather_C2_' + str_interval + '.csv'
df_kier_h_c2.to_csv(str_dirName_h + str_file)
print(df_kier_h_c2.info())
df_kier_h_c2